# СИМА — Полный конвейер обработки рельефа

Демонстрация всех алгоритмов модуля:
1. ЦМР (DTM) — SMRF + IDW + min-Z fallback
2. ЦММ (DSM) — max rasterization
3. Сглаживание (Gaussian / Median)
4. Уклоны и экспозиции
5. TPI (многомасштабный, 270/810/2430 м)
6. Горизонтали (изолинии)
7. Отметки высот

Все растры отображаются в **единой цветовой шкале** для удобства сравнения.

In [ ]:
import sys, os, shutil
from pathlib import Path
import rasterio
import numpy as np
import laspy
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

backend = Path('/Users/sergeyzay/Documents/НЕДРА/СИМА/sima-web/backend')
for pkg in ['packages/sima-dem-core/src', 'packages/sima-dem-ground/src',
            'packages/sima-dem-dsm/src', 'packages/sima-dem-pipeline/src']:
    sys.path.insert(0, str(backend / pkg))

from sima_dem_ground.ground import GroundProcessing, SMRFConfig, FillConfig, RasterOutputConfig
from sima_dem_dsm.dsm import DSMBuilder, DSMConfig
from sima_dem_core.curvature import CurvatureProcessing
from sima_dem_core.raster.smooth import gauss_smooth
from sima_dem_core.raster.median import med_filter
from sima_dem_core.raster.tpi import calculate_tpi, TPIConfig
from sima_dem_core.raster.contours import generate_contours
from sima_dem_core.height import get_every_nth

print('Импорт готов')

In [ ]:
DATASET = 'test'

if DATASET == 'demo':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/pt000100.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/23_04_12_digital_elevation_1-46-315/demo_data/00000100.tif'
    REFERENCE_DSM = None
elif DATASET == 'test':
    LAS_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_ground_TLO.las'
    TIF_PATH = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g.tif'
    REFERENCE_DSM = '/Users/sergeyzay/Documents/НЕДРА/СИМА/test_data/P-42-041-239-g_DSM.tif'

OUTPUT_DIR = str(backend / 'output' / f'notebook_{DATASET}')
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESOLUTION = 1.0
GAUSS_SIGMA = 2.0
GAUSS_WINDOW = 5
MEDIAN_WINDOW = 5
SMOOTHING_METHOD = 'gauss'

crs_source = REFERENCE_DSM if REFERENCE_DSM else TIF_PATH
with rasterio.open(crs_source) as src:
    CRS = src.crs.to_wkt()
print(f'Датасет: {DATASET}, CRS: {CRS[:60]}...')

In [ ]:
if REFERENCE_DSM:
    sys.path.insert(0, str(backend))
    from tests.fixtures.restore_las import restore_absolute_las
    restored = str(Path(OUTPUT_DIR) / 'restored_absolute.las')
    restore_absolute_las(LAS_PATH, REFERENCE_DSM, restored)
    LAS_PATH = restored
    print(f'Восстановлены абсолютные Z: {restored}')

las = laspy.read(LAS_PATH)
cls = np.asarray(las.classification)
print(f'Точек: {len(las.points):,}, классы: {sorted(set(cls))}')
print(f'Z: {np.min(las.z):.1f} – {np.max(las.z):.1f}')

In [ ]:
gp = GroundProcessing(
    output=OUTPUT_DIR, resolution=RESOLUTION, crs=CRS,
    interpolate=True, save_ground_las=False,
    smrf=SMRFConfig(),
    fill=FillConfig(fill_holes=True, max_search_distance=100, fallback_to_min_z=True),
    raster_out=RasterOutputConfig(output_type='idw'),
)
gp.get_raster(LAS_PATH, crs_wkt=CRS)
dtm_path = gp.raster[0]
print(f'DTM: {dtm_path}')

In [ ]:
builder = DSMBuilder(
    output=OUTPUT_DIR, crs=CRS,
    config=DSMConfig(resolution=RESOLUTION, output_type='max',
                     interpolate=True, fill_holes=True),
)
dsm_path = builder.build(LAS_PATH, crs_wkt=CRS)
print(f'DSM: {dsm_path}')

In [ ]:
stem = Path(dtm_path).stem.replace('_dem', '')
smoothed_path = str(Path(OUTPUT_DIR) / (stem + '_dem_smooth.tif'))

if SMOOTHING_METHOD == 'gauss':
    gauss_smooth(dtm_path, smoothed_path,
                 sigma=GAUSS_SIGMA * RESOLUTION, order=0,
                 window_size=GAUSS_WINDOW, fill_holes=True,
                 max_search_distance=100)
elif SMOOTHING_METHOD == 'median':
    shutil.copy2(dtm_path, smoothed_path)
    med_filter(smoothed_path, MEDIAN_WINDOW)

print(f'Сглаженная ({SMOOTHING_METHOD}): {smoothed_path}')

In [ ]:
cp = CurvatureProcessing()
slope_path = cp.calculate_slope(smoothed_path, CRS, RESOLUTION, RESOLUTION, OUTPUT_DIR)
aspect_path = cp.calculate_aspect(smoothed_path, CRS, RESOLUTION, RESOLUTION, OUTPUT_DIR)
print(f'Уклоны: {slope_path}')
print(f'Экспозиции: {aspect_path}')

In [ ]:
tpi_path = calculate_tpi(
    smoothed_path, CRS, OUTPUT_DIR, RESOLUTION, 10.0,
    config=TPIConfig(radii_m=[270, 810, 2430], res=10.0),
)
print(f'TPI: {tpi_path}')

In [ ]:
contour_path = str(Path(OUTPUT_DIR) / 'contours.gpkg')
generate_contours(smoothed_path, interval=2.0, output_path=contour_path, crs_wkt=CRS)
print(f'Горизонтали: {contour_path}')

In [ ]:
ground_las_path = str(Path(OUTPUT_DIR) / 'ground_for_heights.las')
gp2 = GroundProcessing(output=OUTPUT_DIR, resolution=RESOLUTION, crs=CRS,
                        save_ground_las=True)
gp2.get_raster(LAS_PATH, crs_wkt=CRS, out_path=ground_las_path)
heights_path = str(Path(OUTPUT_DIR) / 'heights.geojson')
get_every_nth(ground_las_path, 10, heights_path, CRS)
print(f'Отметки высот: {heights_path}')

In [ ]:
def read_raster(path):
    with rasterio.open(path) as src:
        arr = src.read(1).astype(float)
        if src.nodata is not None:
            arr = np.where(arr == src.nodata, np.nan, arr)
    return arr

dtm = read_raster(dtm_path)
dsm = read_raster(dsm_path)
smoothed = read_raster(smoothed_path)
slope = read_raster(slope_path)
aspect = read_raster(aspect_path)
tpi = read_raster(tpi_path)

z_min = np.nanmin([np.nanmin(dtm), np.nanmin(dsm), np.nanmin(smoothed)])
z_max = np.nanmax([np.nanmax(dtm), np.nanmax(dsm), np.nanmax(smoothed)])
print(f'Общая Z-шкала: {z_min:.1f} – {z_max:.1f} м')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

terrain_norm = mcolors.Normalize(vmin=z_min, vmax=z_max)

rasters = [
    ('DTM (ЦМР)', dtm, 'terrain', terrain_norm),
    ('DSM (ЦММ)', dsm, 'terrain', terrain_norm),
    ('Сглаженная DTM', smoothed, 'terrain', terrain_norm),
    ('Уклоны (°)', slope, 'hot', None),
    ('Экспозиции (°)', aspect, 'hsv', None),
    ('TPI', tpi, 'RdBu_r', None),
]

for ax, (title, data, cmap, norm) in zip(axes.flatten(), rasters):
    im = ax.imshow(data, cmap=cmap, norm=norm)
    ax.set_title(title, fontsize=14)
    plt.colorbar(im, ax=ax, shrink=0.7)

plt.suptitle(f'СИМА — {DATASET} (resolution={RESOLUTION}m, {SMOOTHING_METHOD})', fontsize=16)
plt.tight_layout()
plt.savefig(str(Path(OUTPUT_DIR) / 'overview.png'), dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

if REFERENCE_DSM:
    ref = read_raster(REFERENCE_DSM)
    min_h = min(dtm.shape[0], ref.shape[0])
    min_w = min(dtm.shape[1], ref.shape[1])
    b, r = dtm[:min_h,:min_w], ref[:min_h,:min_w]
    valid = np.isfinite(b) & np.isfinite(r)
    diff = np.where(valid, np.abs(b - r), np.nan)
    rmse = np.sqrt(np.nanmean(diff[valid]**2)) if valid.any() else 0
    mean_ref = np.nanmean(r[valid]) if valid.any() else 0
    rel_err = rmse / mean_ref if mean_ref else 0
    print(f'RMSE: {rmse:.4f} м, Mean: {mean_ref:.4f} м, Ошибка: {rel_err:.4%}')
    print(f'Требование < 5%: {"\u2713 ПРОЙДЕН" if rel_err < 0.05 else "\u2717 НЕ ПРОЙДЕН"}')

    ref_norm = mcolors.Normalize(vmin=z_min, vmax=z_max)
    for ax, title, data in [(axes[0], 'Построено', b), (axes[1], 'Эталон', r)]:
        im = ax.imshow(data, cmap='terrain', norm=ref_norm)
        ax.set_title(title)
        plt.colorbar(im, ax=ax, shrink=0.7)
    im = axes[2].imshow(diff, cmap='Reds')
    axes[2].set_title(f'Разница (RMSE={rmse:.2f} м)')
    plt.colorbar(im, ax=axes[2], shrink=0.7)
else:
    print('Сравнение с эталоном недоступно (demo_data)')
    for ax in axes:
        ax.set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
print(f'Выходные файлы ({OUTPUT_DIR}):')
for f in sorted(Path(OUTPUT_DIR).glob('*')):
    if f.suffix in ('.tif', '.las', '.png', '.gpkg', '.geojson'):
        size = f.stat().st_size / 1024 / 1024
        print(f'  {f.name}  ({size:.1f} MB)')